# Lab: Patterns in ERA5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ml_play/eofs_era5.ipynb)

Unsupervised machine learning finds structure in data without being told what to look for. Here you take 64 years of daily ERA5 and break fields into simple pieces: the patterns the data itself picks out (EOFs).

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q gcsfs zarr cartopy netCDF4

In [ ]:
#@title Setup: imports and ERA5 data
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.special import sph_harm_y

# ERA5 from 1959 to 2022 on a 1.5 degree grid (240 x 121 points), one snapshot per day at 00 UTC.
# This is WeatherBench 2's copy on Google Cloud; no account needed.
era5 = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr",
                    storage_options=dict(token="anon")).sel(time=slice("1959", "2022"))
era5 = era5.sel(time=era5.time.dt.hour == 0)   # one sample per day
era5 = era5.transpose(..., "latitude", "longitude").sortby("latitude")   # stored as (time, lon, lat); put latitude first, south to north
era5

---
## 1. Patterns the data finds: EOFs

Fourier waves are chosen in advance. Empirical orthogonal functions (EOFs, also called principal component analysis) let the data choose. The first EOF is the map pattern that explains the most variance over all the days, the second explains the most of what is left, and so on. Each comes with a time series, its principal component (PC), that says how strongly the pattern is present on each day. The method is never told what El Niño is.

Pick a variable and a box. Longitudes run 0–360 °E.

In [ ]:
# Variables to try (put one in the string below):
#   "sea_surface_temperature"   (~2 min to load)
#   "2m_temperature"            (~5 min)
#   "mean_sea_level_pressure"   (~5 min)
#   500 hPa geopotential is too slow to pull from this copy of ERA5. Use the class file instead:
#     !wget -q -O z500.nc "LINK FROM YOUR INSTRUCTOR"
#     field = xr.open_dataarray("z500.nc")
field = era5["total_precipitation_12hr"]

# Boxes to try:
#   El Niño (sea surface temperature):     latitude=slice(-30, 30), longitude=slice(120, 290)
#   North Pacific (sea level pressure):    latitude=slice(20, 80),  longitude=slice(120, 250)
#   Northern annular mode (geopotential):  latitude=slice(20, 90),  longitude=slice(0, 360), and use monthly means below
#   Global warming (2m temperature):       latitude=slice(-90, 90), longitude=slice(0, 360), use monthly means, skip the detrend line
field = field.sel(latitude=slice(-90, 90), longitude=slice(0, 360)).compute()

anomaly = field.groupby("time.dayofyear") - field.groupby("time.dayofyear").mean()   # remove the seasonal cycle
anomaly = anomaly.resample(time="MS").mean()   # monthly means: uncomment for big boxes, or daily runs out of memory
anomaly = anomaly - xr.polyval(anomaly.time, anomaly.polyfit("time", 1).polyfit_coefficients)   # remove the trend

# The data matrix: one row per day, one column per grid point. Each point is weighted by sqrt(cos(latitude))
# so it counts in proportion to its area. Points with missing values (sea surface temperature over land) are dropped.
stacked = anomaly.stack(point=("latitude", "longitude")).dropna("point")
X = (stacked * np.sqrt(np.cos(np.deg2rad(stacked.latitude)))).values.astype(np.float32)
print(f"{X.shape[0]} times x {X.shape[1]} grid points")

# The EOF analysis itself: a singular value decomposition, X = U S V^T
U, S, VT = np.linalg.svd(X, full_matrices=False)
explained = S ** 2 / np.sum(S ** 2)   # the fraction of the variance each mode explains
pcs = U * S / (U * S).std(axis=0)     # principal components, in units of standard deviations

plt.figure(figsize=(8, 3))
plt.bar(np.arange(1, 21), 100 * explained[:20], color="0.4")
plt.xticks(np.arange(1, 21))
plt.xlabel("mode")
plt.ylabel("% of variance")
plt.title("Variance explained by each mode")
plt.show()

Each map shows the anomaly that goes with one standard deviation of that mode's PC (a regression map), in the variable's units. The sign of a mode is arbitrary: the pattern times the PC is what matters.

In [ ]:
N_MODES = 3

fig = plt.figure(figsize=(14, 3.2 * N_MODES), layout="constrained")
for k in range(N_MODES):
    pc = xr.DataArray(pcs[:, k], coords={"time": anomaly.time})
    pattern = (anomaly * pc).mean("time")   # regression of the anomaly on the PC
    top = float(np.abs(pattern).max())

    ax = fig.add_subplot(N_MODES, 2, 2 * k + 1, projection=ccrs.PlateCarree(central_longitude=180))
    mesh = ax.pcolormesh(pattern.longitude, pattern.latitude, pattern, transform=ccrs.PlateCarree(),
                         cmap="RdBu_r", vmin=-top, vmax=top)
    ax.coastlines(linewidth=0.6)
    fig.colorbar(mesh, ax=ax, shrink=0.8, label=field.attrs.get("units", ""))
    ax.set_title(f"EOF {k + 1} ({explained[k]:.0%})")

    ax = fig.add_subplot(N_MODES, 2, 2 * k + 2)
    ax.plot(anomaly.time, pc, color="black", linewidth=0.5)
    ax.axhline(0, color="0.6", linewidth=0.5)
    ax.set_title(f"PC {k + 1}")
plt.show()

Compare PC 1 with NOAA's Niño 3.4 index, the standard measure of El Niño (sea surface temperature anomaly in 5°S–5°N, 170°W–120°W), month by month.

In [ ]:
URL = "https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data"
table = pd.read_csv(URL, sep=r"\s+", skiprows=1, header=None, index_col=0, nrows=2022 - 1870 + 1)   # one row per year, 1870 to 2022
time = pd.date_range("1870-01", periods=table.size, freq="MS")
nino34_anomaly = xr.DataArray(table.values.ravel(), coords={"time": time})

pc1 = xr.DataArray(pcs[:, 0], coords={"time": anomaly.time}).resample(time="MS").mean()   # monthly, like the NOAA record
nino = nino34_anomaly.sel(time=pc1.time)
pc1 = (pc1 - pc1.mean()) / pc1.std()
nino = (nino - nino.mean()) / nino.std()
r = np.corrcoef(pc1, nino)[0, 1]

plt.figure(figsize=(12, 3.5))
plt.plot(pc1.time, nino, color="tab:red", linewidth=1, label="Niño 3.4 anomaly (NOAA)")
plt.plot(pc1.time, np.sign(r) * pc1, color="black", linewidth=1, label="PC 1" + (" (times -1)" if r < 0 else ""))
plt.ylabel("standard deviations")
plt.title(f"Correlation: {r:.2f}")
plt.legend(loc="upper left")
plt.show()